# GBFC on small-graph — long GPU run (Kaggle)

Gradient-Boosted Front Construction, continuing from the banked best
(−1,828,994, +925 from the leader −1,829,919). Checkpoint-safe: the submission is
re-written **every round**, so a session timeout never loses progress.

**Setup:** Notebook settings → Accelerator → **GPU T4 ×2** (or P100). Internet ON.
Add the project as data: **Add Data → Upload → `torso_project.zip`** (it lands in
`/kaggle/input/<your-dataset>/`). Then Run All.

In [ ]:
import os, glob, zipfile, subprocess, sys
# locate the uploaded zip (in /kaggle/input/*/ or the working dir)
cands = glob.glob('/kaggle/input/**/torso_project.zip', recursive=True) + glob.glob('/kaggle/working/**/torso_project.zip', recursive=True) + glob.glob('torso_project.zip')
assert cands, "Upload torso_project.zip via Add Data first."
zp = cands[0]; print("zip:", zp)
os.makedirs('/kaggle/working/run', exist_ok=True)
with zipfile.ZipFile(zp) as z: z.extractall('/kaggle/working/run')
ROOT = os.path.dirname(glob.glob('/kaggle/working/run/**/tools/gbfc.py', recursive=True)[0]).rsplit('/tools',1)[0]
os.chdir(ROOT); print('cwd:', os.getcwd())

In [ ]:
# deps (Kaggle usually has numba+CUDA and lightgbm; install/verify quietly)
!pip -q install lightgbm numba 2>/dev/null
from numba import cuda; print('CUDA:', cuda.is_available())  # must be True for GPU eval

## Run GBFC — long
`--rounds 4000` with `--round-budget 45` will simply run until the Kaggle session
limit (~12 h), checkpointing `gbfc.json` after every round. It warm-starts from
the bundled −1,828,994 orderings. Watch the `score` / `gap` columns approach
−1,829,919.

In [ ]:
import os; os.environ["PYTHONWARNINGS"]="ignore"
!PYTHONWARNINGS=ignore python3 tools/gbfc.py --problem small-graph --rounds 4000 --round-budget 45 --pop 128 --algo gbfc

## Save the result
The submission is at `submissions/small-graph/gbfc.json` (rewritten every round).
Copy it to `/kaggle/working` so it's saved as notebook output, then download it
and re-score on your machine (`tools/portfolio.py`).

In [ ]:
import shutil
shutil.copy('submissions/small-graph/gbfc.json', '/kaggle/working/gbfc_small.json')
print('saved /kaggle/working/gbfc_small.json — download it from the Output tab')

**Resuming across sessions (to use ~30 h over several runs):** download
`gbfc_small.json`, rename it and drop it back into a fresh project's
`submissions/small-graph/` (or rebuild the warm-start to include it), and re-run —
GBFC loads it and continues. Each session banks whatever it found.

Honest note: small has iterated to −1,828,994 with sharply decelerating gains
(+554 → +40 per run), so 30 h is unlikely to clear −1,829,919 — but it exhausts
the instance, and the submission is always current.